<a href="https://colab.research.google.com/github/jgromero/drl-csic/blob/es/code/stable-baselines/cliffwalking-sb3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DQN para _cliff walking_ con Stable-Baselines3

[**Juan Gómez Romero**](https://ccia.ugr.es/~jgomez)  
Departamento de Ciencias de la Computación e Inteligencia Artificial  
Universidad de Granada  
Este trabajo está bajo licencia [Apache License 2.0](https://choosealicense.com/licenses/apache-2.0/).

---

Ejemplo basado en:
> R.S. Sutton, A.G. Barto (2018) Reinforcement Learning. Capítulos 6.5: Q-learning: control TD _off-policy_, 6.6: _Expected Sarsa_.

> A. Raffin et al. (2021) Stable-Baselines3: Reliable Reinforcement Learning Implementations. _Journal of Machine Learning Research_ 22(268):1-8. Disponible en [GitHub](https://github.com/DLR-RM/stable-baselines3).

_Considere el mundo de cuadrícula ("gridworld") mostrado a la derecha. Es una tarea episódica estándar, sin descuento, con estados inicial y meta, y con las acciones habituales que provocan un movimiento hacia arriba, abajo, derecha e izquierda. La recompensa es -1 en todas las transiciones, salvo en las que llevan a la región marcada como «El Acantilado» ("The Cliff"). Pisar esta región supone una recompensa de -100 y devuelve al agente instantáneamente al estado inicial._

<img src="https://github.com/jgromero/drl-csic/blob/es/img/cliffwalking.png?raw=true" width=500/>

En [cliffwalking-qlearning.ipynb](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/cliffwalking-qlearning.ipynb) resolvimos este problema con nuestra propia implementación de Q-Learning tabular. Aquí usamos la implementación de [DQN](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) de [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) (SB3), una biblioteca de implementaciones fiables y probadas de los algoritmos de aprendizaje por refuerzo profundo (DRL) más comunes, construida sobre PyTorch.

SB3 no incluye métodos tabulares: DQN sustituye la tabla $Q$ por una red neuronal que aproxima $Q(s, a)$. Esto no es necesario para un problema de solo 48 estados, pero muestra que el mismo código puede aplicarse a problemas con espacios de estados mucho mayores (o continuos), donde una tabla no es viable.

## Entorno

In [ ]:
%pip install "gymnasium[toy-text]>=1.1" "stable-baselines3>=2.6" tqdm rich

[Cliff Walking](https://gymnasium.farama.org/environments/toy_text/cliff_walking/) se incluye en [Gymnasium](https://gymnasium.farama.org/) como entorno de ejemplo (y sencillo).

<img src="https://gymnasium.farama.org/_images/cliff_walking.gif"/>

La observación es el índice de la celda en la que se encuentra el agente ($0$ a $47$). SB3 codifica automáticamente esta observación discreta como un vector _one-hot_ de tamaño $48$ antes de pasarla a la red Q.

Caer por el acantilado no termina el episodio, así que una mala política puede caminar indefinidamente. Para evitarlo, limitamos la longitud de los episodios a 100 pasos con el parámetro `max_episode_steps`. También envolvemos el entorno con un [`Monitor`](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html) para registrar la puntuación de cada episodio de entrenamiento.

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

env = Monitor(gym.make("CliffWalking-v1", max_episode_steps=100))
print(env.observation_space)
print(env.action_space)

Para ver cómo actúa un agente, la función `show_episode()` ejecuta un episodio completo y muestra las imágenes generadas por el entorno (`render_mode = "rgb_array"`) como una animación, tanto en Google Colab como en local. La usaremos para comparar el comportamiento del agente antes y después del entrenamiento.

In [ ]:
import io
from PIL import Image
from IPython.display import display, Image as IPImage

def show_episode(env, act, max_t=500):
    """Ejecuta un episodio y lo muestra en el cuaderno como una animación.

    Parámetros
    ==========
        env: entorno creado con render_mode="rgb_array"
        act (callable): función que devuelve la acción para un estado
        max_t (int): número máximo de pasos del episodio
    """
    state, _ = env.reset()
    frames = [Image.fromarray(env.render())]
    score = 0
    for t in range(max_t):
        state, reward, terminated, truncated, _ = env.step(act(state))
        frames.append(Image.fromarray(env.render()))
        score += reward
        if terminated or truncated:
            break

    # guardar las imágenes como un GIF animado y mostrarlo
    gif = io.BytesIO()
    frames[0].save(gif, format='gif', save_all=True, append_images=frames[1:],
                   duration=1000 // env.metadata['render_fps'], loop=0)
    display(IPImage(data=gif.getvalue(), format='gif'))
    print('Puntuación:', score)

Veamos, por ejemplo, un episodio de un agente que elige acciones al azar.

In [ ]:
env_vis = gym.make("CliffWalking-v1", render_mode="rgb_array", max_episode_steps=100)   # entorno para las animaciones

show_episode(env_vis, lambda state: env_vis.action_space.sample(), max_t=100)

## Algoritmo DQN

La clase [`DQN`](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) incluye la red Q, la red objetivo (_target network_), la memoria de experiencias (_replay buffer_) y la estrategia de exploración $\epsilon$-_greedy_. En lugar de un número de episodios, SB3 entrena durante un número dado de pasos de tiempo (`total_timesteps`).

In [ ]:
def plot_scores(scores, title, solved_score=None, window=100):
    """Representa la puntuación de cada episodio de entrenamiento y la media móvil de los últimos `window`.

    Parámetros
    ==========
        scores (list): puntuación de cada episodio de entrenamiento
        title (str): título de la gráfica
        solved_score (float): puntuación media para considerar resuelto el problema (opcional)
        window (int): número de episodios de la media móvil
    """
    scores = np.asarray(scores)
    avg = [np.mean(scores[max(0, i-window+1):i+1]) for i in range(len(scores))]
    plt.plot(np.arange(len(scores)), scores, alpha=0.4, label='Puntuación')
    plt.plot(np.arange(len(scores)), avg, label='Puntuación media (últimos {})'.format(window))
    if solved_score is not None:
        plt.axhline(solved_score, color='gray', linestyle='--', label='Resuelto')
    plt.ylabel('Puntuación')
    plt.xlabel('Episodio nº')
    plt.title(title)
    plt.legend()
    plt.show()

In [ ]:
model = DQN(
    "MlpPolicy",                        # red Q: perceptrón multicapa
    env,
    learning_rate=1e-3,
    batch_size=64,
    buffer_size=50_000,
    learning_starts=1000,               # pasos con acciones aleatorias antes de empezar a aprender
    gamma=0.99,
    target_update_interval=500,         # copiar la red Q en la red objetivo cada 500 pasos
    train_freq=4,                       # actualizar la red Q cada 4 pasos
    exploration_fraction=0.3,           # epsilon decrece linealmente durante el primer 30% del entrenamiento...
    exploration_final_eps=0.05,         # ...hasta este valor
    policy_kwargs=dict(net_arch=[64, 64]),
    seed=0,
    verbose=0,
)

model.learn(total_timesteps=50_000, progress_bar=True)
model.save("cliffwalking-sb3-dqn")   # guardar el agente entrenado

plot_scores(env.get_episode_rewards(), 'DQN')

La política óptima (caminar por el borde del acantilado) obtiene una puntuación de $-13$. Evaluemos el agente con la política _greedy_ (`deterministic=True`). Como el entorno es determinista, un único episodio es suficiente.

In [ ]:
mean_score, _ = evaluate_policy(model, Monitor(gym.make("CliffWalking-v1", max_episode_steps=100)), n_eval_episodes=1, deterministic=True)
print('Puntuación de la política greedy:', mean_score)

La política resultante es la siguiente. La obtenemos pidiendo al modelo la acción _greedy_ (`predict`) en cada uno de los 48 estados.

In [ ]:
def print_policy(actions):
    """Muestra la política como una cuadrícula de flechas (C: acantilado, T: meta).

    Parámetros
    ==========
        actions (array_like): acción de cada uno de los 48 estados (ARRIBA = 0, DERECHA = 1, ABAJO = 2, IZQUIERDA = 3)
    """
    grid = np.array(['↑', '→', '↓', '←'])[np.asarray(actions)].reshape(4, 12)
    grid[3, 1:11] = 'C'   # celdas del acantilado
    grid[3, 11] = 'T'     # meta
    print('\n'.join(' '.join(row) for row in grid))

In [ ]:
states = np.arange(48)
actions, _ = model.predict(states, deterministic=True)

print_policy(actions)

Podemos visualizar los valores de la función de valor de estado $v_\pi$ para la política estimada $\pi$ y compararlos con los valores de la política óptima $v_{\pi_*}$. Aquí los valores $Q(s, a)$ no se leen de una tabla, sino que los calcula la red Q (`model.q_net`).

Nótese que hemos entrenado con $\gamma = 0.99$ en lugar de $\gamma = 1$, por lo que los valores de los estados lejanos a la meta son ligeramente mayores (menos negativos) que los óptimos.

(Se necesita la función `plot_values()` para la visualización.)

In [ ]:
def plot_values(V):
    # redimensionar la función de valor de estado
    V = np.reshape(V, (4,12))
    # representar la función de valor de estado
    fig = plt.figure(figsize=(15,5))
    ax = fig.add_subplot(111)
    im = ax.imshow(V, cmap='cool')
    for (j,i),label in np.ndenumerate(V):
        ax.text(i, j, '{:.1f}'.format(label), ha='center', va='center', fontsize=14)
    plt.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    plt.title('Función de valor de estado')
    plt.show()

__Función de valor de estado de la política aprendida por el agente $v_\pi$__

In [ ]:
import torch

obs, _ = model.policy.obs_to_tensor(states)   # codificación one-hot de los 48 estados
with torch.no_grad():
    Q = model.q_net(obs).cpu().numpy()        # valores Q de los 48 estados, forma (48, 4)

V = Q.max(axis=1)
V[37:] = 0   # las celdas del acantilado y de la meta nunca se visitan
plot_values(V)

__Función de valor de estado de la política óptima $v_{\pi_*}$__

In [ ]:
# Los valores de estado de la política óptima pueden calcularse manualmente
V_opt = np.zeros((4,12))
V_opt[0][:] = -np.arange(3, 15)[::-1]
V_opt[1][:] = -np.arange(3, 15)[::-1] + 1
V_opt[2][:] = -np.arange(3, 15)[::-1] + 2
V_opt[3][0] = -13
plot_values(V_opt)

## Visualizar el agente entrenado

Cargamos el agente entrenado y lo ejecutamos durante un episodio con la política _greedy_ (`deterministic=True`). Las imágenes se muestran como una animación, tanto en Google Colab como en local.

In [ ]:
# cargar el agente entrenado desde `cliffwalking-sb3-dqn.zip`
model_load = DQN.load("cliffwalking-sb3-dqn")

show_episode(env_vis, lambda state: int(model_load.predict(state, deterministic=True)[0]), max_t=100)